# Trial conversion

Conversion rate trends for free trials. Definitions are in `METRICS.md`. Logic lives in `src/`.

In [ ]:
import os
import sys
from pathlib import Path

# Run from the project root so relative paths work.
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from IPython.display import Image, display

from src.plot_trials import (
    OUTPUT_DIR,
    PROCESSED_PATH,
    plot_cancel_timing,
    plot_weekly_conversion,
)
from src.trials_analysis import cancel_timing, matured_trials, weekly_conversion

In [ ]:
df = pd.read_csv(PROCESSED_PATH, parse_dates=["started_at", "ends_at", "outcome_at"])
matured = matured_trials(df)
print(f"{len(df)} trials, {df['is_open'].sum()} open, {len(matured)} matured")
print(f"Conversion rate (matured): {matured['converted'].mean():.1%}")

## Conversion rate by start week

In [ ]:
weekly = weekly_conversion(matured)
weekly

In [ ]:
OUTPUT_DIR.mkdir(exist_ok=True)
plot_weekly_conversion(
    weekly, matured["converted"].mean(), OUTPUT_DIR / "trials_conversion_weekly.png"
)
display(Image(OUTPUT_DIR / "trials_conversion_weekly.png"))

## When trials cancel

In [ ]:
plot_cancel_timing(cancel_timing(matured), OUTPUT_DIR / "trials_cancel_timing.png")
display(Image(OUTPUT_DIR / "trials_cancel_timing.png"))

## Top 3 insights

1. **The conversion rate is about 53.3%, not 50.0%.** Counting only closed trials is biased low, because conversions land on day 14 and cancels come earlier. Use matured trials.
2. **Conversion is flat week to week.** Full weeks range from 51.8% to 54.5%. The 60.2% first week is a partial week with 88 trials, so it is not a trend.
3. **Cancellations are spread evenly across days 4-13.** There are none in the first 3 days and no spike on any one day.